In [2]:
import pandas as pd
import joblib

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler,OrdinalEncoder
from category_encoders import BinaryEncoder
from sklearn.neighbors import KNeighborsRegressor
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import GridSearchCV

df = pd.read_csv(r"C:\Users\Asus\Desktop\Projects\Dataset\cleaned_student_data.csv")

x = df.drop(columns=['student_id','final_exam_score','final_grade'])
y = df['final_exam_score']

x_train,x_test,y_train,y_test = train_test_split(x,y,random_state=42,test_size=0.2)

num_columns = ['study_time_hours','attendance_percent','sleep_hours','previous_grade']
ord_columns = ['parental_education']
nom_columns = ['gender']
bool_columns = ['internet_access','extracurricular_activities','part_time_job']

education_order = ['High School','Bachelors','Masters','PhD']

preprocessor = ColumnTransformer(
    transformers=[
        ('num_scaled',StandardScaler(),num_columns),
        ('ord_encoded',OrdinalEncoder(categories=[education_order]),ord_columns),
        ('nom_encoded',BinaryEncoder(),nom_columns)
    ],remainder='passthrough'
)

pipe = Pipeline(
    steps=[
        ('preprocessing',preprocessor),
        ('regressor',KNeighborsRegressor())
    ]
)

params = {
    'regressor__n_neighbors': range(1, 21),
    'regressor__weights': ['uniform', 'distance'],
    'regressor__p': [1, 2]
}

grid_model = GridSearchCV(pipe,param_grid=params,cv=10,scoring='neg_mean_squared_error')
grid_model.fit(x_train,y_train)

best_model = grid_model.best_estimator_

In [5]:
y_pred = best_model.predict(x_test)

from sklearn.metrics import mean_squared_error
score = mean_squared_error(y_test,y_pred)

print(score)

50.41501093749999


In [1]:
import pandas as pd

df = pd.read_csv(r'C:\Users\Asus\Desktop\Projects\Dataset\cleaned_student_data.csv')

df.sample(5)

,student_id,gender,study_time_hours,attendance_percent,sleep_hours,parental_education,internet_access,extracurricular_activities,part_time_job,previous_grade,final_exam_score,final_grade
784,785,Female,4.6,94.9,5.7,High School,True,True,False,56.6,92.6,A
853,854,Male,5.9,91.9,7.9,Masters,False,True,False,60.4,98.8,A
502,503,Male,4.4,80.3,6.3,High School,False,True,False,76.0,93.3,A
278,279,Male,2.4,98.0,7.4,High School,True,True,False,65.5,94.0,A
28,29,Female,4.0,78.6,8.0,Bachelors,True,False,False,78.2,84.2,B


In [2]:
df.isna().sum()

student_id                    0
gender                        0
study_time_hours              0
attendance_percent            0
sleep_hours                   0
parental_education            0
internet_access               0
extracurricular_activities    0
part_time_job                 0
previous_grade                0
final_exam_score              0
final_grade                   0
dtype: int64

In [5]:
df.groupby('gender')['final_exam_score'].mean()

gender
Female    83.606863
Male      83.477551
Name: final_exam_score, dtype: float64